# DNS Testing Notebook

This notebook demonstrates the DNS tools available in this container.
Run cells with `Shift+Enter`.

## 1. DNS Resolution with dnspython

### A Record Verification

In [ ]:
import dns.resolver

def verify_a_record(domain, expected_ip, dns_server=None):
    """
    Query A record for a domain and verify it matches expected IP.
    Fails if multiple A records are returned.
    """
    try:
        resolver = dns.resolver.Resolver()
        if dns_server:
            resolver.nameservers = [dns_server]
        
        answers = resolver.resolve(domain, 'A')
        
        if len(answers) != 1:
            print(f"\u274c FAIL: Expected 1 A record, got {len(answers)} records")
            for i, answer in enumerate(answers):
                print(f"   Record {i+1}: {answer}")
            return False
        
        actual_ip = str(answers[0])
        
        if actual_ip == expected_ip:
            print(f"\u2705 PASS: {domain} resolves to {actual_ip}")
            return True
        else:
            print(f"\u274c FAIL: {domain} resolves to {actual_ip}, expected {expected_ip}")
            return False
            
    except dns.resolver.NXDOMAIN:
        print(f"\u274c FAIL: Domain {domain} does not exist")
        return False
    except dns.resolver.NoAnswer:
        print(f"\u274c FAIL: No A record found for {domain}")
        return False
    except Exception as e:
        print(f"\u274c ERROR: {str(e)}")
        return False

# Test cases — modify for your needs
test_cases = [
    ("ns1.devries.tv", "74.110.167.35"),
    ("g.root-servers.net", "192.112.36.4"),
]

print("DNS A Record Verification Tests")
print("=" * 40)
all_passed = True
for domain, expected_ip in test_cases:
    result = verify_a_record(domain, expected_ip)
    all_passed &= result
    print()
print("=" * 40)
print("All tests PASSED!" if all_passed else "Some tests FAILED!")

### DNSSEC Validation Test

In [ ]:
import dns.message, dns.query, dns.flags

q = dns.message.make_query("example.com", "A", want_dnssec=True)
r = dns.query.udp(q, "9.9.9.9", timeout=2)

if r.flags & dns.flags.AD:
    print("AD flag set — DNSSEC validation passed")
else:
    print("AD flag not set — DNSSEC validation failed")

## 2. CLI Tools — dig, kdig, q

In [ ]:
# Standard dig query
!dig example.com A +short

In [ ]:
# kdig with DNS-over-TLS to Cloudflare
!kdig -d @1.1.1.1 +tls-ca example.com A

In [ ]:
# q — DNS-over-HTTPS query to Google
!q example.com A @https://dns.google/dns-query

## 3. DNSSEC Chain Visualization with dnsviz

In [ ]:
# Probe DNSSEC chain for example.com and render as a PNG
!dnsviz probe example.com > /tmp/dnsviz-probe.json 2>/dev/null
!dnsviz graph -Tpng -o /tmp/dnsviz-example.png < /tmp/dnsviz-probe.json 2>/dev/null

from IPython.display import Image, display
display(Image(filename='/tmp/dnsviz-example.png'))

## 4. DNS Performance Benchmarking

In [ ]:
# dnspyre — benchmark 100 queries against multiple resolvers
import subprocess, json, pandas as pd, matplotlib.pyplot as plt

resolvers = {
    "Cloudflare": "1.1.1.1",
    "Google": "8.8.8.8",
    "Quad9": "9.9.9.9",
}

results = []
for name, ip in resolvers.items():
    print(f"Benchmarking {name} ({ip})...")
    proc = subprocess.run(
        ["dnspyre", "-n", "100", "-c", "5", "--json",
         "--server", ip, "example.com"],
        capture_output=True, text=True, timeout=30
    )
    if proc.returncode == 0:
        data = json.loads(proc.stdout)
        for timing in data:
            results.append({
                "resolver": name,
                "latency_ms": timing.get("latencyStats", {}).get("avgMs", 0),
                "success_rate": timing.get("codes", {}).get("NOERROR", 0) / max(timing.get("totalRequests", 1), 1) * 100,
            })

if results:
    df = pd.DataFrame(results)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    df.plot.bar(x="resolver", y="latency_ms", ax=ax1, legend=False, color="steelblue")
    ax1.set_title("Average Latency (ms)")
    ax1.set_ylabel("ms")
    df.plot.bar(x="resolver", y="success_rate", ax=ax2, legend=False, color="seagreen")
    ax2.set_title("Success Rate (%)")
    ax2.set_ylabel("%")
    ax2.set_ylim(0, 105)
    plt.tight_layout()
    plt.show()
else:
    print("No results — check that dnspyre is available")

## 5. BIND Zone File Validation

In [ ]:
# Write a sample zone file and validate it
zone_content = """\
$TTL 3600
@  IN  SOA  ns1.example.com. admin.example.com. (
           2024010101  ; serial
           3600        ; refresh
           900         ; retry
           604800      ; expire
           86400 )     ; minimum

@       IN  NS   ns1.example.com.
@       IN  NS   ns2.example.com.
@       IN  A    93.184.216.34
www     IN  CNAME @
ns1     IN  A    93.184.216.10
ns2     IN  A    93.184.216.11
"""

with open("/tmp/example.zone", "w") as f:
    f.write(zone_content)

!named-checkzone example.com /tmp/example.zone

## 6. Exporting This Notebook to PDF

Run the cell below to export this notebook (with all current outputs) to PDF:

In [ ]:
!jupyter nbconvert --to pdf /workspace/notebooks/dns-examples.ipynb --output /workspace/notebooks/dns-examples.pdf 2>&1 | tail -3